In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import os
import re
import glob
from multiprocessing import Pool, cpu_count
from loguru import logger

In [ ]:
# Define the path to synchronized data directory
script_dir = os.path.dirname(os.path.abspath('__file__'))
filtered_dir = os.path.join(script_dir, '..', '..', 'filtered')
metadata_dir = os.path.join(script_dir, '..', '..', 'metadata')

# 2. Correlation validations

In [ ]:
def correlation_check(
    df_synced: pd.DataFrame,
    column_name_x: str,
    column_name_y: str,
    line_graph_start,
    line_graph_end,
    mean_ratio: bool = False,
    # --- Speed filter ---
    min_speed: float = 0.0,
    speed_column: str | None = None,   # if None, auto-pick STW then SOG
    stw_qid: str = DEFAULT_STW_QID,
    sog_qid: str = DEFAULT_SOG_QID,
    # --- Metadata for pretty naming ---
    metadata: pd.DataFrame | str | None = None,  # DataFrame or CSV path
    metadata_qid_col: str = "qid_mapping",
    metadata_quantity_col: str = "quantity_name",
    metadata_source_col: str = "source_name",
    metadata_unit_col: str = "unit",
    # --- Plot tuning ---
    scatter_s: float = 1,
    scatter_alpha: float = 0.03,
    # --- Line plot indexing (ONLY affects the line plot panel) ---
    line_index: bool = True,
    line_index_mode: str = "base100",  # "base100" | "minmax" | "zscore"
):
    """
    Output:
      - Matplotlib figure with two side-by-side plots:
          (1) scatter: x vs y using ALL rows (after optional speed filter)
              + trendline + stats box (r, N, fit, mean-ratio)
          (2) line plot: x and y vs time within [line_graph_start, line_graph_end]
              (after optional speed filter), optionally indexed for comparability
      - Returns: (fig, (ax_scatter, ax_line))

    Notes:
      - min_speed filter uses `speed_column` if provided; else prefers STW then SOG.
      - QID->label lookup triggers only if ':' is in the column name (your requirement).
      - Line indexing is applied ONLY to the line plot (panel 2), scatter stays raw units.
    """

    # ---- Basic checks ----
    if column_name_x not in df_synced.columns:
        raise KeyError(f"column_name_x='{column_name_x}' not in df_synced.columns")
    if column_name_y not in df_synced.columns:
        raise KeyError(f"column_name_y='{column_name_y}' not in df_synced.columns")

    # ---- Load/prepare metadata mapping ----
    meta_map = None
    if metadata is not None:
        meta_df = pd.read_csv(metadata) if isinstance(metadata, str) else metadata

        needed = {metadata_qid_col, metadata_quantity_col, metadata_source_col, metadata_unit_col}
        missing = needed - set(meta_df.columns)
        if missing:
            raise ValueError(
                f"metadata is missing columns: {sorted(missing)}. "
                f"Got: {list(meta_df.columns)}"
            )

        meta_df = meta_df.copy()
        meta_df[metadata_qid_col] = meta_df[metadata_qid_col].astype(str).str.strip()

        meta_map = meta_df.set_index(metadata_qid_col)[
            [metadata_quantity_col, metadata_source_col, metadata_unit_col]
        ].to_dict(orient="index")

    # Token pattern: grabs chunks like "2::0::6::1_1::1::0::2::0_1::0::1::0_8"
    token_pattern = re.compile(r"[0-9:]+(?:_[0-9:]+)*")

    def _clean_unit(u: str) -> str:
        return str(u).replace("Â", "").strip()

    def _extract_qid(col: str) -> str | None:
        if meta_map is None:
            return None
        c = str(col).strip()
        if c in meta_map:
            return c

        # Try to extract a token that matches a metadata key
        for m in token_pattern.finditer(c):
            tok = m.group(0)
            if "::" in tok and tok in meta_map:
                return tok
        return None

    def _pretty_label(col: str) -> str:
        c = str(col).strip()

        # Only trigger lookup if ':' is present (your requirement)
        if ":" not in c or meta_map is None:
            return c

        qid = _extract_qid(c)
        if qid is None:
            return c

        rec = meta_map[qid]
        qname = str(rec.get(metadata_quantity_col, "")).strip()
        sname = str(rec.get(metadata_source_col, "")).strip()
        unit = _clean_unit(rec.get(metadata_unit_col, ""))

        if unit:
            return f"{qname} ({sname}) [{unit}]"
        return f"{qname} ({sname})"

    label_x = _pretty_label(column_name_x)
    label_y = _pretty_label(column_name_y)

    # ---- Min speed filtering ----
    df_plot = df_synced
    if min_speed is None:
        min_speed = 0.0
    min_speed = float(min_speed)

    if min_speed > 0.0:
        if speed_column is None:
            if stw_qid in df_plot.columns:
                speed_column = stw_qid
            elif sog_qid in df_plot.columns:
                speed_column = sog_qid
            else:
                raise KeyError(
                    "min_speed > 0 but no speed_column provided and neither default STW nor SOG QID exists in df_synced."
                )
        if speed_column not in df_plot.columns:
            raise KeyError(f"speed_column='{speed_column}' not in df_synced.columns")

        speed_vals = pd.to_numeric(df_plot[speed_column], errors="coerce")
        df_plot = df_plot.loc[speed_vals >= min_speed].copy()

    # ---- Scatter data (ALL rows, ignoring time) ----
    x = pd.to_numeric(df_plot[column_name_x], errors="coerce")
    y = pd.to_numeric(df_plot[column_name_y], errors="coerce")
    mask_xy = x.notna() & y.notna()

    x_vals = x[mask_xy].to_numpy(dtype="float64", copy=False)
    y_vals = y[mask_xy].to_numpy(dtype="float64", copy=False)
    mask_finite = np.isfinite(x_vals) & np.isfinite(y_vals)
    x_vals = x_vals[mask_finite]
    y_vals = y_vals[mask_finite]
    n = int(x_vals.size)

    ratio = None
    if mean_ratio:
        mean_x = float(np.mean(x_vals)) if n else float("nan")
        mean_y = float(np.mean(y_vals)) if n else float("nan")
        ratio = (mean_x / mean_y) if (np.isfinite(mean_y) and mean_y != 0.0) else float("nan")
        print(f"Mean ratio (mean({column_name_x}) / mean({column_name_y})) = {ratio:.6g}")

    r = float("nan")
    if n >= 2:
        sx = float(np.std(x_vals))
        sy = float(np.std(y_vals))
        if sx > 0 and sy > 0:
            r = float(np.corrcoef(x_vals, y_vals)[0, 1])

    slope = float("nan")
    intercept = float("nan")
    if n >= 2:
        x_mean = float(np.mean(x_vals))
        y_mean = float(np.mean(y_vals))
        x_var = float(np.mean((x_vals - x_mean) ** 2))
        if x_var > 0:
            cov_xy = float(np.mean((x_vals - x_mean) * (y_vals - y_mean)))
            slope = cov_xy / x_var
            intercept = y_mean - slope * x_mean

    # ---- Line-plot data (time window) ----
    start = pd.to_datetime(line_graph_start)
    end = pd.to_datetime(line_graph_end)
    if start > end:
        raise ValueError("line_graph_start must be <= line_graph_end")

    if isinstance(df_plot.index, pd.DatetimeIndex):
        df_time = df_plot.sort_index().loc[start:end, [column_name_x, column_name_y]]
        time_index = df_time.index
    elif "utc_timestamp" in df_plot.columns or "timestamp" in df_plot.columns:
        ts_col = "utc_timestamp" if "utc_timestamp" in df_plot.columns else "timestamp"
        ts = pd.to_datetime(df_plot[ts_col])
        df_tmp = df_plot.assign(_ts=ts).sort_values("_ts")
        df_time = df_tmp[(df_tmp["_ts"] >= start) & (df_tmp["_ts"] <= end)][["_ts", column_name_x, column_name_y]]
        time_index = df_time["_ts"]
    else:
        raise ValueError("df_synced must have a DatetimeIndex or a 'utc_timestamp'/'timestamp' column for the line plot.")

    # ---- Line indexing helper (applies ONLY to panel 2) ----
    def _index_series(s: pd.Series, mode: str) -> np.ndarray:
        s = pd.to_numeric(s, errors="coerce").astype("float64")

        if mode == "zscore":
            arr = s.to_numpy()
            mu = float(np.nanmean(arr))
            sd = float(np.nanstd(arr))
            if np.isfinite(sd) and sd > 0:
                return ((s - mu) / sd).to_numpy()
            return np.full(len(s), np.nan)

        if mode == "minmax":
            arr = s.to_numpy()
            a = float(np.nanmin(arr))
            b = float(np.nanmax(arr))
            if np.isfinite(a) and np.isfinite(b) and b > a:
                return ((s - a) / (b - a)).to_numpy()
            return np.full(len(s), np.nan)

        # default: base100
        valid = s[np.isfinite(s)]
        if valid.empty:
            return np.full(len(s), np.nan)
        base = float(valid.iloc[0])
        if np.isfinite(base) and base != 0.0:
            return (s / base * 100.0).to_numpy()
        return np.full(len(s), np.nan)

    # ---- Prepare line-series values ----
    if line_index:
        x_line_vals = _index_series(df_time[column_name_x], line_index_mode)
        y_line_vals = _index_series(df_time[column_name_y], line_index_mode)
        label_x_line = f"{label_x} (indexed)"
        label_y_line = f"{label_y} (indexed)"
        ylab = "Indexed value"
    else:
        x_line_vals = pd.to_numeric(df_time[column_name_x], errors="coerce").to_numpy()
        y_line_vals = pd.to_numeric(df_time[column_name_y], errors="coerce").to_numpy()
        label_x_line = label_x
        label_y_line = label_y
        ylab = ""

    # ---- Plotting ----
    fig, (ax_scatter, ax_line) = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

    ax_scatter.scatter(
        x_vals, y_vals,
        s=scatter_s, alpha=scatter_alpha, marker=".", linewidths=0,
        rasterized=True,
    )
    ax_scatter.set_xlabel(label_x)
    ax_scatter.set_ylabel(label_y)
    ax_scatter.set_title(f"Scatter (min speed: {min_speed:g})" if min_speed > 0 else "Scatter (all rows)")

    if np.isfinite(slope) and np.isfinite(intercept) and n >= 2:
        if n >= 100:
            x1, x2 = np.quantile(x_vals, [0.01, 0.99])
        else:
            x1, x2 = float(np.min(x_vals)), float(np.max(x_vals))
        x_fit = np.array([x1, x2], dtype="float64")
        y_fit = intercept + slope * x_fit
        ax_scatter.plot(x_fit, y_fit, linewidth=2.0, color="tab:orange", alpha=0.9)

    stats_lines = [
        f"N = {n:,}",
        f"Pearson r = {r:.3f}" if np.isfinite(r) else "Pearson r = NaN",
        f"Fit: y = {slope:.4g}x + {intercept:.4g}" if (np.isfinite(slope) and np.isfinite(intercept)) else "Fit: NaN",
    ]
    if mean_ratio:
        stats_lines.append(
            f"Mean ratio = {ratio:.4g}" if (ratio is not None and np.isfinite(ratio)) else "Mean ratio = NaN"
        )

    ax_scatter.text(
        0.02, 0.98, "\n".join(stats_lines),
        transform=ax_scatter.transAxes,
        va="top", ha="left", fontsize=9,
        bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="0.3", alpha=0.85),
    )

    ax_line.plot(time_index, x_line_vals, label=label_x_line, linewidth=0.8)
    ax_line.plot(time_index, y_line_vals, label=label_y_line, linewidth=0.8)
    ax_line.set_xlabel("Time")
    if ylab:
        ax_line.set_ylabel(ylab)
    suffix = f" (indexed: {line_index_mode})" if line_index else ""
    ax_line.set_title(f"Time window: {start} to {end}{suffix}")
    ax_line.legend()

    return fig, (ax_scatter, ax_line)
